# Лабораторна робота № 5
## Географічні дані OpenStreetMap: від тегів до маршруту

**Інтелектуальний аналіз та візуалізація даних** · 4 години · 4 бали

---

### Постановка

Ви — аналітик у туристичному застосунку. Завдання: для заданого міста
**відібрати десять найвартісніших історико-культурних об'єктів** і
**спланувати піший маршрут**, який починається на залізничному вокзалі,
обходить усі десять і повертається на вокзал.

Слово «найвартісніші» в даних не записане. Критерій ви побудуєте самі —
і захистите його.

Потім те саме для **тридцяти** об'єктів. Результат цієї, більшої задачі
стане базою порівняння в **лабораторній роботі № 8**, де ту саму задачу
розв'язуватимуть генетичним та мурашиним алгоритмами.

### Що здаєте

1. Цей ноутбук із виконаними комірками (усі виводи збережені).
2. Файл `lr5_результати.json` — результати для ЛР8.
3. Карту `lr5_маршрут.html`.
4. Звіт у кінці ноутбука: відповіді на сім питань.

### Що оцінюється

| критерій | 0 | 2 | 3 | 4 |
|---|---|---|---|---|
| Відбір об'єктів | теги взято навмання | фільтр працює | фільтр обґрунтовано | показано, що відкинуто і чому |
| Критерій рейтингу | немає | є формула | ваги пояснено | порівняно з альтернативним критерієм |
| Маршрут | не будується | будується | перевірено коректність | пояснено різницю методів числом |
| Карта | немає | точки на карті | маршрут по ребрах графа | видно різницю з прямими відрізками |
| Звіт | немає | формально | з числами | з висновком, якого не було в завданні |

---
## 0. Ваш варіант

Впишіть прізвище та номер у списку групи. Місто підтягнеться само.

In [33]:
STUDENT_NAME = 'Боднар Денис Васильович'     # напр. 'Коваленко Іван Петрович'
LIST_NUMBER  = 1      # номер у списку групи

CITIES = [
    'Львів', 'Київ', 'Харків', 'Одеса', 'Дніпро',
    'Чернівці', 'Івано-Франківськ', 'Тернопіль', 'Ужгород', 'Луцьк',
    'Рівне', 'Хмельницький', 'Вінниця', 'Житомир', 'Чернігів',
    'Черкаси', 'Полтава', 'Суми', 'Кропивницький', 'Запоріжжя',
    'Миколаїв', 'Кривий Ріг', 'Кам\u2019янець-Подільський', 'Мукачево',
    'Коломия', 'Дрогобич', 'Біла Церква', 'Ніжин', 'Остріг', 'Умань',
]

assert STUDENT_NAME and LIST_NUMBER > 0, 'впишіть прізвище та номер у списку'
CITY = CITIES[(LIST_NUMBER - 1) % len(CITIES)] + ', Україна'
print('варіант %d: %s' % (LIST_NUMBER, CITY))

варіант 1: Львів, Україна


> Якщо викладач призначив вам інше місто — просто перепишіть `CITY` вручну
> і зазначте це у звіті.

In [34]:
import importlib.util, subprocess, sys
REQUIRED = ['osmnx', 'geopandas', 'shapely', 'pyproj', 'pyarrow',
            'folium', 'mapclassify', 'h3', 'networkx']
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print('встановлюю:', ', '.join(missing), '...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('усі пакети на місці')

усі пакети на місці


In [35]:
import warnings; warnings.filterwarnings('ignore')
import json, time, os, collections
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import folium
from xyzservices import providers as xyz

BASEMAP = xyz.Esri.WorldGrayCanvas   # CARTO з 2026 р. вимагає ключ API
pd.set_option('display.width', 200)
print('geopandas', gpd.__version__)

geopandas 1.2.0


---
## 1. Два способи дістати дані

Робоча програма вимагає порівняти два способи доступу **за обсягом переданих
даних і за часом**:

* **живий запит** — `osmnx` питає Overpass API просто зараз;
* **знімок** — ті самі дані, збережені у файли GeoParquet, читаються з диска.

Нижче — живий запит. Запустіть його **один раз**: він збереже теку зі знімком,
і всі наступні запуски читатимуть з неї.

> Overpass — безкоштовний сервіс і обмежує частоту. Якщо запит не пройшов —
> зачекайте хвилину й повторіть. Якщо не проходить взагалі, візьміть
> резервний знімок у викладача і вкажіть це у звіті.

In [36]:
import osmnx as ox

OUT = 'osm_' + CITY.split(',')[0].strip().lower().replace(' ', '_')
os.makedirs(OUT, exist_ok=True)
ox.settings.use_cache = True
ox.settings.cache_folder = os.path.join(OUT, 'cache')
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True

TOURISM_OK = ['museum', 'attraction', 'gallery', 'artwork', 'viewpoint']

def cache_bytes():
    import pathlib
    p = pathlib.Path(ox.settings.cache_folder)
    return sum(f.stat().st_size for f in p.glob('*.json')) if p.exists() else 0

print('тека знімка:', OUT)

тека знімка: osm_львів


### Дві налаштовані дрібниці, без яких запит не пройде

Це не формальності — кожна з них ламає збір по-своєму, і жодна не схожа
на свою справжню причину.

**Представитись.** Overpass відповідає **HTTP 406** клієнтові, який прийшов
із типовим заголовком `python-requests`. `osmnx` надсилає свій `User-Agent`
сам, але якщо ви звертаєтесь до сервісу напряму — надсилайте теж.
Рядок має бути **лише латиницею**: HTTP-заголовки передаються в latin-1,
і кирилиця кладе запит помилкою `UnicodeEncodeError` ще до відправлення.

**Обрати машину.** За іменем `overpass-api.de` стоїть кілька серверів, і
частина з них відмовляє у з’єднанні. `osmnx` закріплює одну адресу на
всю сесію — навмисно, щоб не порушувати облік слотів. Якщо закріпиться та,
що відмовляє, впаде весь збір. Тому адресу обираємо самі, вже перевірену.

In [37]:
import socket

USER_AGENT = ('LvivPolytechnic-SK26-coursework/1.0 '
              '(https://lpnu.ua; student lab assignment)')
assert USER_AGENT.isascii(), 'HTTP-заголовки — latin-1, кирилиця тут не працює'
ox.settings.http_user_agent = USER_AGENT


def pin_working_ip(hostname, port=443, timeout=8):
    """Закріпити для цього імені адресу, яка справді відповідає."""
    try:
        infos = socket.getaddrinfo(hostname, port, socket.AF_INET, socket.SOCK_STREAM)
    except Exception as e:
        print('не вдалось резолвити %s: %s' % (hostname, type(e).__name__))
        return None
    ips = sorted({i[4][0] for i in infos})
    good = []
    for ip in ips:
        s = socket.socket(); s.settimeout(timeout)
        try:
            s.connect((ip, port)); good.append(ip)
        except Exception:
            pass
        finally:
            s.close()
    print('%s: машин %d, відповідають %d %s' % (hostname, len(ips), len(good), good))
    if not good:
        return None
    original = socket.gethostbyname
    socket.gethostbyname = lambda h: good[0] if h == hostname else original(h)
    print('закріплено', good[0])
    return good[0]


pin_working_ip('overpass-api.de')

overpass-api.de: машин 2, відповідають 2 ['162.55.144.139', '65.109.112.52']
закріплено 162.55.144.139


'162.55.144.139'

> Якщо жодна машина не відповіла — виконайте комірку ще раз за хвилину,
> або візьміть у викладача резервний знімок і зазначте це у звіті.
> Інші дзеркала перелічені на сторінці
> [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API);
> адресу можна підставити у `ox.settings.overpass_url`.

### Завдання 1.1

Допишіть функцію `fetch`: вона має зберегти шар у `.parquet`, а якщо файл
уже є — прочитати його, **не звертаючись до мережі**. Поверніть також,
скільки МБ пішло мережею і скільки секунд це зайняло.

Підказка щодо `clean`: у OSM одна колонка може містити і число, і список
(ребро, склеєне з кількох шляхів). Parquet вимагає один тип на колонку,
тому неоднорідну колонку треба звести до тексту **цілком**.

In [38]:
def clean(gdf):
    g = gdf.reset_index()
    for c in g.columns:
        if c == 'geometry':
            continue
        boxed = g[c].map(lambda v: isinstance(v, (list, dict, set, tuple))).any()
        if boxed or g[c].dtype == object:
            g[c] = g[c].map(
                lambda v: None if v is None or (isinstance(v, float) and pd.isna(v))
                else (json.dumps(v, ensure_ascii=False)
                      if isinstance(v, (list, dict, set, tuple)) else str(v))
            ).astype('string')
    return g


COST = {}
LIVE_LOG = os.path.join(OUT, 'cost_live.json')


def _log_live(name, rec):
    """Запам’ятати вартість ЖИВОГО запиту у файл. Під час повторного запуску
    COST уже описує знімок, і без цього журналу порівнювати стало б нема з чим."""
    try:
        log = json.load(open(LIVE_LOG, encoding='utf-8')) if os.path.exists(LIVE_LOG) else {}
    except Exception:
        log = {}
    log[name] = rec
    with open(LIVE_LOG, 'w', encoding='utf-8') as f:
        json.dump(log, f, ensure_ascii=False, indent=1)


def fetch(name, getter):
    """Повертає шар. Якщо файл уже збережено — читає з диска, мережі не торкається."""
    path = os.path.join(OUT, name + '.parquet')
    if os.path.exists(path) and os.path.getsize(path) > 0:
        t0 = time.time()
        g = gpd.read_parquet(path)
        COST[name] = {'спосіб': 'знімок', 'записів': len(g),
                      'передано_МБ': 0.0,
                      'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                      'секунд': round(time.time() - t0, 3)}
        return g

    # ----- живий запит -----
    t0, b0 = time.time(), cache_bytes()          # 1) що було ДО виклику
    g = clean(getter())                           # 2) запит + приведення типів
    g.to_parquet(path, index=False)               #    збереження знімка
    g = gpd.read_parquet(path)                    #    читаємо назад: що збережено, те й використовуємо
    COST[name] = {'спосіб': 'живий запит', 'записів': len(g),   # 3) вартість
                  'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                  'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                  'секунд': round(time.time() - t0, 1)}
    _log_live(name, COST[name])
    return g

In [39]:
boundary = fetch('boundary', lambda: ox.geocode_to_gdf(CITY))
poly = boundary.geometry.iloc[0]
print('межа міста отримана, площа %.1f км²'
      % (boundary.to_crs(32635).area.iloc[0] / 1e6))

межа міста отримана, площа 148.4 км²


In [40]:
heritage = fetch('heritage', lambda: ox.features_from_polygon(
    poly, {'historic': True, 'tourism': TOURISM_OK}))
stations = fetch('stations', lambda: ox.features_from_polygon(
    poly, {'railway': ['station', 'halt'], 'building': ['train_station']}))
amenities = fetch('amenities', lambda: ox.features_from_polygon(
    poly, {'amenity': ['cafe', 'restaurant', 'drinking_water', 'toilets']}))

# Мережа приходить двома таблицями одразу, тому їй окрема гілка.
NODES_P = os.path.join(OUT, 'road_nodes.parquet')
EDGES_P = os.path.join(OUT, 'road_edges.parquet')

if os.path.exists(NODES_P) and os.path.exists(EDGES_P):
    t0 = time.time()
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'знімок', 'записів': len(nodes) + len(edges),
                           'передано_МБ': 0.0,
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 3)}
else:
    t0, b0 = time.time(), cache_bytes()
    G_raw = ox.graph_from_polygon(poly, network_type='walk')
    nodes, edges = ox.convert.graph_to_gdfs(G_raw)
    clean(nodes).to_parquet(NODES_P, index=False)
    clean(edges).to_parquet(EDGES_P, index=False)
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'живий запит', 'записів': len(nodes) + len(edges),
                           'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 1)}
    _log_live('мережа walk', COST['мережа walk'])

print('вузлів %d | ребер %d' % (len(nodes), len(edges)))

# --- охоронна перевірка: це справді ПІШОХІДНА мережа? ---
# Автомобільна мережа теж читається без помилок, і далі все «працює».
_hw = edges['highway'].astype(str)
N_PED = int(_hw.str.contains('footway|pedestrian|steps|path').sum())
print('пішохідних ребер (footway/pedestrian/steps/path): %d з %d (%.0f%%)'
      % (N_PED, len(edges), 100 * N_PED / len(edges)))
assert N_PED > 0, ('У мережі немає жодного пішохідного ребра — це автомобільна мережа '
                   '(так буває з резервним знімком). Видаліть road_nodes.parquet і '
                   'road_edges.parquet з теки %s і запустіть комірку ще раз.' % OUT)


вузлів 50591 | ребер 145084
пішохідних ребер (footway/pedestrian/steps/path): 75138 з 145084 (52%)


### Завдання 1.2

Зведіть `COST` у таблицю і дайте відповідь у звіті: **у скільки разів
читання знімка швидше за живий запит** і **скільки мегабайтів ви не передали
мережею** при повторному запуску.

Контрольна ознака правильності: при **другому** запуску ноутбука всі рядки
мають мати спосіб «знімок», а `передано_МБ` — нуль.

In [41]:
display(pd.DataFrame(COST).T)


def live_cost():
    """Вартість живого запиту: з журналу ноутбука або з manifest.json скрипта fetch_city_osm.py."""
    rows = {}
    mf = os.path.join(OUT, 'manifest.json')
    if os.path.exists(mf):
        try:
            m = json.load(open(mf, encoding='utf-8'))
        except Exception:
            m = {}
        for k, v in m.items():
            if not (isinstance(v, dict) and 'секунд' in v and 'передано_МБ' in v):
                continue
            if k in ('road_nodes', 'road_edges'):          # мережа — одним записом
                r = rows.setdefault('мережа walk', {'секунд': 0.0, 'передано_МБ': 0.0})
                r['секунд'] += v['секунд']
                r['передано_МБ'] = max(r['передано_МБ'], v['передано_МБ'])
            else:
                rows[k] = {'секунд': v['секунд'], 'передано_МБ': v['передано_МБ']}
    if os.path.exists(LIVE_LOG):
        for k, v in json.load(open(LIVE_LOG, encoding='utf-8')).items():
            rows[k] = {'секунд': v['секунд'], 'передано_МБ': v['передано_МБ']}
    return rows


LIVE = live_cost()
common = [k for k in COST if k in LIVE and COST[k]['спосіб'] == 'знімок']
SPEEDUP = MB_SAVED = T_LIVE = T_SNAP = None
if common:
    T_LIVE = sum(LIVE[k]['секунд'] for k in common)
    T_SNAP = max(sum(COST[k]['секунд'] for k in common), 1e-3)
    MB_SAVED = sum(LIVE[k]['передано_МБ'] for k in common)
    SPEEDUP = T_LIVE / T_SNAP
    cmp_ = pd.DataFrame({'живий запит, с': {k: LIVE[k]['секунд'] for k in common},
                         'знімок, с': {k: COST[k]['секунд'] for k in common},
                         'передано живим, МБ': {k: LIVE[k]['передано_МБ'] for k in common}})
    display(cmp_)
    print('усі шари разом: живий запит %.1f с, знімок %.3f с  ->  знімок швидший у %.0f раз'
          % (T_LIVE, T_SNAP, SPEEDUP))
    print('не передано мережею при повторному запуску: %.2f МБ' % MB_SAVED)
else:
    print('Живих запитів у журналі немає (або це перший запуск, коли всі шари «живі»).')
    print('Запустіть ноутбук ще раз — у таблиці мають бути лише «знімок» і нулі.')

,спосіб,записів,передано_МБ,файл_МБ,секунд
boundary,знімок,1,0.0,0.028,0.035
heritage,знімок,726,0.0,0.221,0.022
stations,знімок,26,0.0,0.046,0.012
amenities,знімок,1307,0.0,0.189,0.017
мережа walk,знімок,195675,0.0,10.072,0.15


,"живий запит, с","знімок, с","передано живим, МБ"
boundary,3.4,0.035,0.045
heritage,110.0,0.022,0.486
stations,99.5,0.012,0.031
amenities,118.7,0.017,0.448
мережа walk,60.7,0.150,22.599


усі шари разом: живий запит 392.3 с, знімок 0.236 с  ->  знімок швидший у 1662 раз
не передано мережею при повторному запуску: 23.61 МБ


> **Чому пішохідна мережа, а не автомобільна.** У старому місті музеї стоять
> за сотню метрів один від одного, а автомобільних вузлів там одиниці. Кілька
> пам'яток сядуть на **один** вузол графа, і матриця віддалей скаже, що між
> ними нуль метрів. Задача розв'яжеться, відповідь буде неправильною, і
> повідомлення про помилку не з'явиться. У кроці 4 ви це перевірите явно.

---
## 2. Хто претендує на звання пам'ятки

У OpenStreetMap немає тега «визначна пам'ятка». Є два сімейства:

* `historic=*` — `castle`, `church`, `monument`, `memorial`, `ruins`, `tower`, `yes`…
* `tourism=*` — `museum`, `gallery`, `attraction`, `artwork`, `viewpoint`

Перетин неповний: музей у сучасній будівлі має лише `tourism`, руїни фортеці —
лише `historic`. Тому беремо **об'єднання**.

📖 <https://wiki.openstreetmap.org/wiki/Map_Features>

### Завдання 2.1

Відберіть кандидатів і **назвіть у звіті, скільки об'єктів ви відкинули й чому**.

In [42]:
import re


def txt(df, col):
    """Колонка як текст без порожніх клітинок. Немає колонки — порожній ряд."""
    if col not in df.columns:
        return pd.Series('', index=df.index, dtype='string')
    return df[col].astype('string').fillna('').str.strip()


# правило 1: теги — ОБ’ЄДНАННЯ historic і tourism, а не перетин
has_hist = txt(heritage, 'historic') != ''
has_tour = txt(heritage, 'tourism').isin(TOURISM_OK)
mask = has_hist | has_tour
# правило 2: маршрут не може містити точку «якась історична будівля»
has_name = txt(heritage, 'name') != ''

cand = heritage[mask & has_name].reset_index(drop=True)
cand['uid'] = cand['element'].astype(str).str[0] + cand['id'].astype(str)   # n123 / w456 / r789

rej_tags = heritage[~mask]
rej_name = heritage[mask & ~has_name]
N_RAW, N_BADTAG, N_NONAME, N_CAND = len(heritage), len(rej_tags), len(rej_name), len(cand)
N_NONAME_WIKI = int(((txt(rej_name, 'wikidata') != '') | (txt(rej_name, 'wikipedia') != '')).sum())

print('усього у шарі heritage :', N_RAW)
print('підходять за тегами    :', int(mask.sum()))
print('з них мають name       :', N_CAND)

print()
print('Що відкинуто і чому:')
display(pd.DataFrame({'правило': ['1. теги не з historic/tourism-переліку', '2. немає name'],
                      'відкинуто': [N_BADTAG, N_NONAME],
                      'частка від шару, %': [round(100 * N_BADTAG / N_RAW, 1),
                                             round(100 * N_NONAME / N_RAW, 1)]}).set_index('правило'))
print('Безіменні за типом (що саме ми викидаємо):')
kind = (txt(rej_name, 'historic').where(txt(rej_name, 'historic') != '', 'tourism=' + txt(rej_name, 'tourism')))
print(kind.value_counts().head(8).to_string())
print()
print('Безіменних, але з wikidata/wikipedia: %d  — відомі об’єкти, у яких просто не вписали name' % N_NONAME_WIKI)
if N_NONAME_WIKI:
    cols = [c for c in ['uid', 'historic', 'tourism', 'wikidata'] if c in rej_name.columns]
    tmp = rej_name.copy()
    tmp['uid'] = tmp['element'].astype(str).str[0] + tmp['id'].astype(str)
    display(tmp[((txt(tmp, 'wikidata') != '') | (txt(tmp, 'wikipedia') != ''))][cols].head(5))

усього у шарі heritage : 726
підходять за тегами    : 726
з них мають name       : 540

Що відкинуто і чому:


,відкинуто,"частка від шару, %"
правило,,
1. теги не з historic/tourism-переліку,0,0.0
2. немає name,186,25.6


Безіменні за типом (що саме ми викидаємо):
historic
tourism=artwork       53
wayside_cross         31
memorial              31
tourism=viewpoint     23
tourism=attraction     9
tomb                   8
ruins                  5
heritage               4

Безіменних, але з wikidata/wikipedia: 9  — відомі об’єкти, у яких просто не вписали name


,historic,tourism,wikidata
630,<NA>,gallery,Q123494199
633,building,<NA>,Q12118181
670,water_well,attraction,<NA>
696,citywalls,<NA>,Q12134978
697,citywalls,<NA>,Q12134978


In [43]:
assert 'cand' in dir(), 'спершу виконайте завдання 2.1'
print('=== historic ===');  print(cand['historic'].value_counts().head(10).to_string())
print()
print('=== tourism ===');   print(cand['tourism'].value_counts().head(10).to_string())
print()
print('типи геометрії:', cand.geometry.geom_type.value_counts().to_dict())

=== historic ===
historic
memorial         174
tomb              53
wayside_cross     38
yes               21
house             18
vehicle           13
monument          12
ruins             10
railway_car       10
railway            7

=== tourism ===
tourism
museum        51
artwork       50
attraction    47
viewpoint      6
gallery        5

типи геометрії: {'Point': 421, 'Polygon': 111, 'LineString': 8}


> **Перевірте себе.** Серед типів геометрії мають бути і `Point`, і `Polygon`.
> Якщо самі лише полігони — ви відфільтрували зайве: пам'ятники, меморіали та
> оглядові точки в OSM є **вузлами**, у них немає будівлі.

### Якщо кандидатів менше тридцяти

Невелике місто може дати 15–20 об'єктів. Тоді:

1. розширте перелік `tourism` (наприклад, додайте `theatre`, `place_of_worship`);
2. або зменште `N_LARGE` до того, що є;

і **обов'язково зазначте це у звіті** разом із числом.

---
## 3. Критерій «найкращі»

Ключовий крок роботи. Стовпця «цінність» у даних немає. Його треба
сконструювати і захистити.

Сигнали, які є в OSM:

| сигнал | що означає |
|---|---|
| `wikidata` / `wikipedia` | про об'єкт є стаття — хтось сторонній визнав його вартим опису |
| кількість колонок `name:*` | назву переклали N мовами — перекладають те, що шукають іноземці |
| `tourism=museum/gallery` | призначений для відвідування |
| `historic` заповнений | має історичну природу |
| площа полігона | розмір — **найслабший сигнал**: великим буває і склад |

### Завдання 3.1

Побудуйте числовий рейтинг. Вимоги:

* використати **щонайменше три** різні сигнали;
* звести різнорідні величини до спільної шкали (підказка: `rank(pct=True)`);
* **пояснити у звіті, чому саме такі ваги**.

In [44]:
metric = cand.to_crs(32635)
cand['area_m2'] = np.where(metric.geometry.geom_type.isin(['Polygon', 'MultiPolygon']),
                           metric.area.values, 0.0)

# сигнал 1: є стаття (wikidata або wikipedia) — визнання ззовні OSM
cand['wiki'] = ((txt(cand, 'wikidata') != '') | (txt(cand, 'wikipedia') != '')).astype(int)

# сигнал 2: у скількох мовах є назва. Рахуємо name:xx (2-3 літери), без рідної name:uk;
# name:uk-Latn, name:signed і т. п. сюди не потрапляють
lang_cols = [c for c in cand.columns if re.fullmatch(r'name:[a-z]{2,3}', c) and c != 'name:uk']
cand['n_lang'] = (cand[lang_cols].astype('string').fillna('')
                  .apply(lambda s: s.str.strip() != '').sum(axis=1)).astype(int) if lang_cols else 0

# сигнал 3: тип «для відвідування»; сигнал 4: має історичну природу; сигнал 5: площа
cand['visit'] = txt(cand, 'tourism').isin(['museum', 'gallery', 'attraction', 'viewpoint']).astype(int)
cand['hist'] = (txt(cand, 'historic') != '').astype(int)

# різнорідні величини — на спільну шкалу 0..1
cand['lang_r'] = cand['n_lang'].rank(pct=True)
cand['area_r'] = cand['area_m2'].rank(pct=True)

WEIGHTS = {'wiki': 0.35, 'lang_r': 0.25, 'visit': 0.20, 'hist': 0.10, 'area_r': 0.10}
assert abs(sum(WEIGHTS.values()) - 1) < 1e-9
cand['score'] = sum(w * cand[c] for c, w in WEIGHTS.items())

print('кандидатів   :', len(cand))
print('мають wiki   :', int(cand['wiki'].sum()))
print('мовні колонки:', len(lang_cols), '| мають >=1 переклад:', int((cand['n_lang'] > 0).sum()))
print('ваги         :', WEIGHTS)

кандидатів   : 540
мають wiki   : 97
мовні колонки: 13 | мають >=1 переклад: 256
ваги         : {'wiki': 0.35, 'lang_r': 0.25, 'visit': 0.2, 'hist': 0.1, 'area_r': 0.1}


### Завдання 3.2 — обов'язкове порівняння

Побудуйте **другий** рейтинг за найпростішим критерієм — площею будівлі —
і порахуйте, скільки об'єктів збігається в перших десяти.

Це головне питання роботи: обидва списки отримані з тих самих даних, тим
самим кодом, без жодної помилки. Якщо вони різні — поясніть у звіті, що це
означає для слова «найкращі».

In [45]:
N_SMALL, N_LARGE = 10, 30
POOL = 120      # запас для чистки дублікатів: вона може вилучити кілька об’єктів

# однозначний порядок: за рівності балів — більше мов, потім назва та id (відтворюваність для ЛР8)
ranked = (cand.sort_values(['score', 'n_lang', 'name', 'id'], ascending=[False, False, True, True])
          .reset_index(drop=True))
top_raw = ranked.head(N_LARGE).copy()          # топ-30 ДО чистки дублікатів
top_naive = (cand.sort_values(['area_m2', 'name', 'id'], ascending=[False, True, True])
             .head(N_SMALL).reset_index(drop=True))

overlap_raw = len(set(top_raw.head(N_SMALL)['uid']) & set(top_naive['uid']))
print('перетин топ-10 за вашим критерієм і за площею (до чистки): %d з %d' % (overlap_raw, N_SMALL))
display(pd.DataFrame({'за вашим критерієм': list(top_raw.head(N_SMALL)['name']),
                      'за площею': list(top_naive['name'])}))

перетин топ-10 за вашим критерієм і за площею (до чистки): 2 з 10


,за вашим критерієм,за площею
0,Палац Потоцьких,Личаківський цвинтар
1,Вежа Корнякта,Марсове поле
2,Національний музей у Львові,Меморіал львівських орлят
3,Королівський арсенал,"Австрійський форт (Fort ""Zjawienska Gora"")"
4,Кам'яниця Корнякта,Меморіал Українським Січовим стрільцям та воїн...
5,Львівський музей історії релігії,Австрійський форт
6,Львівський Державний Природознавчий Музей,Львівський музей історії релігії
7,"Арсенал, Львівський державний історичний музей...",Національний музей у Львові
8,Порохова Вежа,Польський військовий меморіал
9,Львівська національна галерея мистецтв,Будинок вчених


### Завдання 3.3 — дублікати

OSM наповнюють різні люди. Той самий музей трапляється двічі під різними
назвами, а вежа буває окремим об'єктом усередині палацу. Знайдіть у `top_score`
пари, що стоять ближче **60 метрів** одна до одної, і вирішіть, що з ними робити.

Рішення (лишити / злити / видалити) опишіть у звіті.

In [46]:
from difflib import SequenceMatcher


def close_pairs(gdf, limit_m=60):
    p = gdf.to_crs(32635).geometry.representative_point()
    XY = np.c_[p.x.values, p.y.values]
    D = np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    out = [(i, j, D[i, j]) for i in range(len(XY)) for j in range(i + 1, len(XY))
           if D[i, j] < limit_m]
    return sorted(out, key=lambda t: t[2])


# Перевірка самої функції на завідомо близьких точках (контрольна ознака з методички)
_t = gpd.GeoDataFrame({'n': ['a', 'b', 'c']},
                      geometry=gpd.points_from_xy([24.0300, 24.0301, 24.0400], [49.84, 49.84, 49.84]),
                      crs='EPSG:4326')
assert [(i, j) for i, j, _ in close_pairs(_t)] == [(0, 1)], 'close_pairs працює неправильно'


def _norm(s):
    return re.sub(r'[^0-9a-zа-яіїєґ]+', '', str(s).lower())


def same_object(a, b):
    """Чи це один і той самий об’єкт? Повертає причину або None.
    Три ознаки: однаковий wikidata; схожі назви; один об’єкт лежить усередині іншого."""
    wa, wb = ('' if pd.isna(a.get('wikidata')) else str(a['wikidata'])), ('' if pd.isna(b.get('wikidata')) else str(b['wikidata']))
    if wa and wa == wb:
        return 'однаковий wikidata'
    na, nb = _norm(a['name']), _norm(b['name'])
    if na and nb and (min(len(na), len(nb)) >= 4 and (na in nb or nb in na)
                      or SequenceMatcher(None, na, nb).ratio() >= 0.8):
        return 'схожі назви'
    ga, gb = a['_gm'], b['_gm']
    if ga.geom_type in ('Polygon', 'MultiPolygon') and gb.geom_type in ('Polygon', 'MultiPolygon'):
        inter = ga.intersection(gb).area
        if inter > 0.5 * min(ga.area, gb.area):
            return 'один контур усередині іншого'
    elif ga.geom_type in ('Polygon', 'MultiPolygon') and gb.geom_type == 'Point':
        if ga.buffer(2).contains(gb):
            return 'точка всередині контуру'
    elif gb.geom_type in ('Polygon', 'MultiPolygon') and ga.geom_type == 'Point':
        if gb.buffer(2).contains(ga):
            return 'точка всередині контуру'
    return None


ranked['_gm'] = ranked.to_crs(32635).geometry.values
ranked['_pm'] = ranked['_gm'].map(lambda g: g.representative_point())


def dist_m(a, b):
    return a['_pm'].distance(b['_pm'])


# 1) що відбувається в топ-30 ДО чистки
pairs = close_pairs(top_raw, 60)
rows = []
for i, j, d in pairs:
    a, b = ranked.iloc[i], ranked.iloc[j]          # top_raw — перші 30 рядків ranked
    why = same_object(a, b)
    rows.append({'об’єкт А': a['name'], 'об’єкт Б': b['name'], 'відстань, м': round(d, 1),
                 'висновок': ('дублікат: ' + why) if why else 'різні об’єкти поруч',
                 'рішення': 'злити (лишити вище в рейтингу)' if why else 'лишити обидва'})
PAIRS_DF = pd.DataFrame(rows)
print('пар ближче 60 м у топ-%d до чистки: %d' % (N_LARGE, len(pairs)))
if len(PAIRS_DF):
    display(PAIRS_DF)

# 2) чистка: іде за рейтингом зверху вниз; об’єкт, що дублює вже взятий, пропускається,
#    його місце займає наступний із рейтингу
kept, dropped = [], []
for _, row in ranked.head(POOL).iterrows():
    dup = None
    for k in kept:
        if dist_m(row, k) < 60:
            r = same_object(row, k)
            if r:
                dup = (k['name'], r); break
    if dup:
        dropped.append((row['name'], dup[0], dup[1]))
    else:
        kept.append(row)
    if len(kept) == N_LARGE:
        break

top_score = pd.DataFrame(kept).drop(columns=['_gm', '_pm']).reset_index(drop=True)
top_score = gpd.GeoDataFrame(top_score, geometry='geometry', crs=cand.crs)
top10 = top_score.head(N_SMALL)
N_DUP_DROPPED = len(dropped)
overlap = len(set(top10['uid']) & set(top_naive['uid']))

print()
print('вилучено дублікатів при доборі топ-%d: %d' % (N_LARGE, N_DUP_DROPPED))
for nm, of, why in dropped[:10]:
    print('   - «%s» дублює «%s» (%s)' % (nm, of, why))
print('перетин топ-10 за критерієм і за площею (після чистки): %d з %d' % (overlap, N_SMALL))

left = close_pairs(top_score, 60)
print('пар ближче 60 м, що лишились (різні об’єкти): %d' % len(left))
for i, j, d in left[:8]:
    print('   %-45s | %-45s %5.1f м' % (top_score['name'].iloc[i][:45], top_score['name'].iloc[j][:45], d))

пар ближче 60 м у топ-30 до чистки: 6


,об’єкт А,об’єкт Б,"відстань, м",висновок,рішення
0,Львівський історичний музей,Музей скла,20.2,різні об’єкти поруч,лишити обидва
1,Кам'яниця Корнякта,Львівський історичний музей,26.0,різні об’єкти поруч,лишити обидва
2,Кам'яниця Корнякта,Музей скла,44.3,різні об’єкти поруч,лишити обидва
3,Львівський історичний музей,Музей міста,50.5,різні об’єкти поруч,лишити обидва
4,Музей Михайла Грушевського,Музей Івана Франка,55.8,різні об’єкти поруч,лишити обидва
5,Палац Потоцьких,Музей мистецтва давньої української книги,58.8,різні об’єкти поруч,лишити обидва



вилучено дублікатів при доборі топ-30: 0
перетин топ-10 за критерієм і за площею (після чистки): 2 з 10
пар ближче 60 м, що лишились (різні об’єкти): 6
   Львівський історичний музей                   | Музей скла                                     20.2 м
   Кам'яниця Корнякта                            | Львівський історичний музей                    26.0 м
   Кам'яниця Корнякта                            | Музей скла                                     44.3 м
   Львівський історичний музей                   | Музей міста                                    50.5 м
   Музей Михайла Грушевського                    | Музей Івана Франка                             55.8 м
   Палац Потоцьких                               | Музей мистецтва давньої української книги      58.8 м


---
## 4. Старт: вокзал, і прив'язка точок до графа

In [47]:
print('станції у шарі:', len(stations))
print(stations[['name', 'railway', 'building']].dropna(how='all').head(12).to_string())

станції у шарі: 26
                 name  railway building
0               Рудно  station     <NA>
1            5-й Парк     halt     <NA>
2    Львів-Коновальця     halt     <NA>
3   Львів-Персенківка  station     <NA>
4          Зубрівська     halt     <NA>
5             Сонячна  station     <NA>
6             Паркова  station     <NA>
7      Львів-Клепарів  station     <NA>
8       Княгині Ольги     halt     <NA>
9    Львів-Городоцька     halt     <NA>
10              Рясна     halt     <NA>
11            Скнилів  station     <NA>


In [48]:
# Головний вокзал: найбільше балів за назву, потім — контур будівлі (стоїть біля входу),
# потім — тег railway=station. «Приміський вокзал» і «ст. Підзамче» мають поступитись.
cs = stations.copy()
nm_ = txt(cs, 'name').str.lower()
cs['бали'] = (3 * nm_.str.contains('головн').astype(int)
              + 1 * nm_.str.contains('вокзал').astype(int)
              + 1 * cs.geometry.geom_type.isin(['Polygon', 'MultiPolygon']).astype(int)
              + 1 * (txt(cs, 'railway') == 'station').astype(int))
cs = cs.sort_values('бали', ascending=False, kind='mergesort')
print(cs[['name', 'railway', 'building', 'бали']].dropna(how='all').head(6).to_string())

station = cs.head(1).copy()
assert len(station) == 1 and station['бали'].iloc[0] >= 3, 'не знайшов головний вокзал серед станцій'
print('\nстарт:', station['name'].iloc[0], '|', station.geometry.iloc[0].geom_type)

                 name  railway       building  бали
21    Головний вокзал     <NA>  train_station     5
18     Львів-Головний  station           <NA>     4
20  Приміський вокзал     <NA>  train_station     2
24               <NA>  station            yes     2
0               Рудно  station           <NA>     1
3   Львів-Персенківка  station           <NA>     1

старт: Головний вокзал | Polygon


In [49]:
G = nx.DiGraph()
for t in nodes.itertuples():
    G.add_node(t.osmid, x=t.x, y=t.y)
for t in edges.itertuples():
    L = float(t.length)
    if G.has_edge(t.u, t.v):
        G[t.u][t.v]['length'] = min(G[t.u][t.v]['length'], L)
    else:
        G.add_edge(t.u, t.v, length=L)
print('граф: %d вузлів, %d ребер' % (G.number_of_nodes(), G.number_of_edges()))

NODES_M = nodes.to_crs(32635)
NODE_XY = np.c_[NODES_M.geometry.x.values, NODES_M.geometry.y.values]
NODE_ID = nodes['osmid'].values

def snap(points_gdf):
    """Найближчий вузол графа до кожної точки. Рахуємо в метричній CRS."""
    pmet = points_gdf.to_crs(32635)
    ids, dist = [], []
    for p in pmet.geometry:
        d2 = (NODE_XY[:, 0] - p.x) ** 2 + (NODE_XY[:, 1] - p.y) ** 2
        k = int(np.argmin(d2))
        ids.append(NODE_ID[k]); dist.append(float(np.sqrt(d2[k])))
    return np.array(ids), np.array(dist)

def make_points(top):
    names = ['СТАРТ: ' + str(station['name'].iloc[0])] + list(top['name'])
    geoms = [station.geometry.iloc[0].representative_point()] + \
            [g.representative_point() for g in top.geometry]
    return gpd.GeoDataFrame({'name': names}, geometry=geoms, crs='EPSG:4326')

граф: 50591 вузлів, 143716 ребер


### Завдання 4.1 — обов'язкова перевірка

Напишіть `check_snap`, яка повідомляє:

* скільки точок і скільки **різних** вузлів вони зайняли;
* медіану й максимум відстані «точка → вузол»;
* якщо вузлів менше, ніж точок — **назви точок, що злиплись**.

**Контрольна ознака:** на пішохідній мережі різних вузлів має бути **стільки ж,
скільки точок**. Якщо менше — ви взяли автомобільну мережу або місто надто мале;
зафіксуйте це у звіті, бо далі такі пари отримають нульову віддаль.

In [50]:
SNAP = {}


def check_snap(points, ids, dist, label):
    ids = list(ids); dist = np.asarray(dist, dtype=float)
    n, uniq = len(ids), len(set(ids))
    print('%s: точок %d, різних вузлів %d' % (label, n, uniq))
    print('  відстань точка → вузол: медіана %.1f м, максимум %.1f м'
          % (np.median(dist), dist.max()))
    groups = []
    if uniq < n:
        by = collections.defaultdict(list)
        for k, nid in enumerate(ids):
            by[nid].append(str(points['name'].iloc[k]))
        groups = [v for v in by.values() if len(v) > 1]
        print('  УВАГА: %d точок злиплись на спільних вузлах — між ними віддаль 0 м:' % (n - uniq))
        for g in groups:
            print('     · ' + '  +  '.join(g))
    else:
        print('  усі точки сіли на різні вузли — можна рахувати')
    far = [(str(points['name'].iloc[k]), float(dist[k])) for k in range(n) if dist[k] > 100]
    if far:
        print('  далі ніж 100 м від найближчого вузла:', ', '.join('%s (%.0f м)' % t for t in far))
    SNAP[label] = {'точок': n, 'вузлів': uniq, 'медіана_м': float(np.median(dist)),
                   'максимум_м': float(dist.max()), 'злиплі': groups, 'далеко': far}
    return SNAP[label]

---
## 5. Матриця віддалей

Дві матриці: **пряма лінія** і **дорогою**. Їхнє відношення — коефіцієнт
обходу — показує, наскільки пряма лінія бреше.

### Завдання 5.1

In [51]:
def road_matrix(ids):
    """Матриця найкоротших віддалей графом між заданими вузлами."""
    k = len(ids)
    D = np.full((k, k), np.inf)
    for i, s in enumerate(ids):
        lengths = nx.single_source_dijkstra_path_length(G, s, weight='length')
        for j, t in enumerate(ids):
            D[i, j] = lengths.get(t, np.inf)
    np.fill_diagonal(D, 0.0)
    return D


def straight_matrix(points):
    """Прямі віддалі в метрах. Рахуємо в метричній CRS, а не в градусах."""
    p = points.to_crs(32635).geometry
    XY = np.c_[p.x.values, p.y.values]
    return np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))


def detour_factor(D, E):
    """Медіана відношення дорога / пряма по скінченних парах i != j.
    Пари з D = 0 (точки злиплись на одному вузлі) і E = 0 у відношення не беремо:
    вони дали б 0 або ділення на нуль і не про обхід."""
    m = np.isfinite(D) & (D > 0) & (E > 0)
    np.fill_diagonal(m, False)
    return float(np.median(D[m] / E[m]))

**Контрольні ознаки — звіртесь, не підганяючи:**

* коефіцієнт обходу для пішохідної мережі у місті зазвичай **1,2–1,6**;
* якщо у вас рівно **1,0** — ви порівнюєте матрицю саму із собою;
* якщо **більше 2** — найімовірніше взято автомобільну мережу з односторонніми
  вулицями, або місто має річку/залізницю, яку нема де перетнути. Друге —
  цікавий результат, перше — помилка. Розберіться, що саме у вас, і напишіть це.

Якщо серед віддалей трапились `inf` — частина графа не зв'язана з рештою.
Не замінюйте `inf` великим числом мовчки: спершу з'ясуйте, які це точки.

In [52]:
pts10 = make_points(top10)
ids10, dist10 = snap(pts10)
check_snap(pts10, ids10, dist10, 'мала задача')

t0 = time.time(); D10 = road_matrix(ids10); print('матриця за %.3f с' % (time.time() - t0))
E10 = straight_matrix(pts10)
print('недосяжних пар    :', int(np.isinf(D10).sum()))
print('асиметрія         : %.0f м' % float(np.max(np.abs(D10 - D10.T))))
print('коефіцієнт обходу : %.2f' % detour_factor(D10, E10))

мала задача: точок 11, різних вузлів 11
  відстань точка → вузол: медіана 19.0 м, максимум 47.2 м
  усі точки сіли на різні вузли — можна рахувати
матриця за 0.514 с
недосяжних пар    : 0
асиметрія         : 0 м
коефіцієнт обходу : 1.32


> **Асиметрія.** $D_{ij} \\neq D_{ji}$ — не помилка: односторонній рух робить
> шлях туди і назад різним. На пішохідній мережі асиметрія мала, на
> автомобільній — помітна. Подивіться на своє число і поясніть його у звіті.

---
## 6. Задача комівояжера

| метод | що робить | складність |
|---|---|---|
| найближчий сусід | щоразу йде в найближчу невідвідану точку | $O(n^2)$ |
| 2-opt | розвертає ділянки готового маршруту, поки коротшає | $O(n^2)$ за прохід |
| Held–Karp | динамічне програмування за підмножинами — **точний** | $O(2^n n^2)$ часу, $O(2^n n)$ пам'яті |

Held–Karp дано готовим: він не є метою роботи, але без нього ви не дізнаєтесь,
**наскільки** помиляються перші два.

### Завдання 6.1 — реалізуйте перші два методи

In [53]:
def tour_length(D, tour):
    return float(sum(D[tour[i], tour[i + 1]] for i in range(len(tour) - 1)))

def nearest_neighbour(D):
    """Маршрут 0 -> ... -> 0. Щоразу йдемо в найближчу невідвідану точку."""
    k = len(D)
    left, tour, cur = set(range(1, k)), [0], 0
    while left:
        nxt = min(sorted(left), key=lambda j: D[cur, j])     # sorted — щоб рівні віддалі давали однаковий результат
        tour.append(nxt); left.remove(nxt); cur = nxt
    tour.append(0)
    return tour

def two_opt(D, tour):
    """Беремо маршрут і розвертаємо ділянки, поки довжина зменшується.
    Довжину після розвороту рахуємо повністю, а не за формулою «чотири ребра»:
    матриця дорогою трохи несиметрична, і для неї швидка формула була б неточною."""
    best = list(tour)
    best_len = tour_length(D, best)
    improved = True
    while improved:
        improved = False
        for i in range(1, len(best) - 2):          # позиції 0 і останню (старт) не чіпаємо
            for j in range(i + 1, len(best) - 1):
                new = best[:i] + best[i:j + 1][::-1] + best[j + 1:]
                L = tour_length(D, new)
                if L < best_len - 1e-9:
                    best, best_len, improved = new, L, True
    return best

In [54]:
def held_karp(D):
    """Точний розв'язок. Вузол 0 — старт і фініш. ДАНО, писати не треба."""
    k = len(D); N = k - 1; INF = float('inf')
    dp = [[INF] * N for _ in range(1 << N)]
    par = [[-1] * N for _ in range(1 << N)]
    for j in range(N):
        dp[1 << j][j] = D[0, j + 1]
    for S in range(1 << N):
        for j in range(N):
            cur = dp[S][j]
            if cur == INF or not (S >> j) & 1:
                continue
            for t in range(N):
                if (S >> t) & 1:
                    continue
                S2 = S | (1 << t); v = cur + D[j + 1, t + 1]
                if v < dp[S2][t]:
                    dp[S2][t] = v; par[S2][t] = j
    full = (1 << N) - 1
    bj = min(range(N), key=lambda j: dp[full][j] + D[j + 1, 0])
    best = dp[full][bj] + D[bj + 1, 0]
    path = []; S = full; j = bj
    while j != -1:
        path.append(j + 1); pj = par[S][j]; S ^= (1 << j); j = pj
    return [0] + path[::-1] + [0], float(best)

### Самоперевірка

Ці твердження мають виконатись для **будь-якого** міста. Якщо хоч одне впало —
шукайте помилку в коді, а не в даних.

In [55]:
def check_tour(tour, k, label):
    assert tour[0] == 0 and tour[-1] == 0, '%s: маршрут має починатись і закінчуватись на старті' % label
    assert sorted(tour[:-1]) == list(range(k)), '%s: кожна точка рівно один раз' % label
    print('%s: структура маршруту правильна (%d точок)' % (label, k))

nn10 = nearest_neighbour(D10)
op10 = two_opt(D10, nn10)
ex10, L_opt = held_karp(D10)

for t, lbl in [(nn10, 'найближчий сусід'), (op10, '2-opt'), (ex10, 'Held-Karp')]:
    check_tour(t, len(ids10), lbl)

assert tour_length(D10, op10) <= tour_length(D10, nn10) + 1e-6, '2-opt не може бути гіршим за вхідний маршрут'
assert L_opt <= tour_length(D10, op10) + 1e-6, 'точний розв\u2019язок не може бути гіршим за наближений'
print()
print('усі перевірки пройдено')

найближчий сусід: структура маршруту правильна (11 точок)
2-opt: структура маршруту правильна (11 точок)
Held-Karp: структура маршруту правильна (11 точок)

усі перевірки пройдено


In [56]:
rows = []
for name, tour in [('найближчий сусід', nn10), ('2-opt', op10), ('Held-Karp (точний)', ex10)]:
    L = tour_length(D10, tour)
    rows.append({'метод': name, 'довжина, м': round(L),
                 'гірше оптимуму, %': round(100 * (L / L_opt - 1), 2)})
display(pd.DataFrame(rows).set_index('метод'))

print()
print('оптимальний порядок:')
for step, i in enumerate(ex10):
    print('  %2d. %s' % (step, pts10['name'].iloc[i]))
print()
print('довжина: %.2f км' % (L_opt / 1000))

,"довжина, м","гірше оптимуму, %"
метод,,
найближчий сусід,8516,6.83
2-opt,8181,2.62
Held-Karp (точний),7972,0.00



оптимальний порядок:
   0. СТАРТ: Головний вокзал
   1. Національний музей у Львові
   2. Львівський Державний Природознавчий Музей
   3. Кам'яниця Корнякта
   4. Львівський музей історії релігії
   5. Королівський арсенал
   6. Вежа Корнякта
   7. Порохова Вежа
   8. Арсенал, Львівський державний історичний музей старовинної зброї
   9. Палац Потоцьких
  10. Львівська національна галерея мистецтв
  11. СТАРТ: Головний вокзал

довжина: 7.97 км


### Завдання 6.2 — велика задача

Спершу оцініть, чи можливий тут точний розв'язок. Порахуйте для
$n = 11, 16, 21, 26, 31$ розмір таблиці Held–Karp ($2^{n-1}\\cdot(n-1)$ клітин)
і скільки це гігабайтів при 8 байтах на клітину.

In [57]:
rows = []
for n in [11, 16, 21, 26, 31]:
    cells = 2 ** (n - 1) * (n - 1)              # клітин у таблиці g(S, j)
    ops = 2 ** (n - 1) * (n - 1) ** 2           # приблизно стільки операцій
    rows.append({'точок n': n, 'клітин': '%.1e' % cells,
                 'пам’ять, ГБ (8 байт/клітина)': '%.3g' % (cells * 8 / 1e9),
                 'операцій': '%.1e' % ops})
display(pd.DataFrame(rows).set_index('точок n'))

HK_CELLS31 = 2 ** 30 * 30
HK_GB31 = HK_CELLS31 * 8 / 1e9
HK_OPS31 = 2 ** 30 * 30 ** 2
print('n = 31 (вокзал + 30 об’єктів): %.1e клітин = %.0f ГБ, ~%.1e операцій' % (HK_CELLS31, HK_GB31, HK_OPS31))
print('кожні +5 точок множать потребу у %d разів' % round((2 ** 30 * 30) / (2 ** 25 * 25)))

,клітин,"пам’ять, ГБ (8 байт/клітина)",операцій
точок n,,,
11,1.0e+04,8.19e-05,1.0e+05
16,4.9e+05,0.00393,7.4e+06
21,2.1e+07,0.168,4.2e+08
26,8.4e+08,6.71,2.1e+10
31,3.2e+10,258,9.7e+11


n = 31 (вокзал + 30 об’єктів): 3.2e+10 клітин = 258 ГБ, ~9.7e+11 операцій
кожні +5 точок множать потребу у 38 разів


Подивіться на останній рядок і напишіть у звіті, **чому саме** точний розв'язок
на 30 точках недоступний. Слово «довго» відповіддю не є.

In [58]:
pts30 = make_points(top_score)
ids30, dist30 = snap(pts30)
check_snap(pts30, ids30, dist30, 'велика задача')

D30 = road_matrix(ids30)
E30 = straight_matrix(pts30)
print('коефіцієнт обходу: %.2f' % detour_factor(D30, E30))

nn30 = nearest_neighbour(D30)
t0 = time.time(); op30 = two_opt(D30, nn30); t_op30 = time.time() - t0
check_tour(op30, len(ids30), '2-opt на 30 точках')

L_nn30, L_op30 = tour_length(D30, nn30), tour_length(D30, op30)
print()
print('найближчий сусід : %8.0f м' % L_nn30)
print('2-opt            : %8.0f м  за %.3f с  (коротше на %.1f%%)'
      % (L_op30, t_op30, 100 * (1 - L_op30 / L_nn30)))
print()
print('ЦЕ — база порівняння для лабораторної роботи № 8.')

велика задача: точок 31, різних вузлів 30
  відстань точка → вузол: медіана 19.0 м, максимум 47.2 м
  УВАГА: 1 точок злиплись на спільних вузлах — між ними віддаль 0 м:
     · Кам'яниця Корнякта  +  Львівський історичний музей
коефіцієнт обходу: 1.22
2-opt на 30 точках: структура маршруту правильна (31 точок)

найближчий сусід :    38235 м
2-opt            :    35856 м  за 0.003 с  (коротше на 6.2%)

ЦЕ — база порівняння для лабораторної роботи № 8.


---
## 7. Просторова індексація: H3

H3 — ієрархічна сітка шестикутників. Рівень 9 — приблизно 200 м по ребру.

📖 <https://h3geo.org/docs/core-library/restable/>

### Завдання 7.1

Розкладіть **усіх** кандидатів по комірках H3 рівня 9 і побудуйте картограму
щільності. На тій самій карті покажіть межу міста і точки вашого маршруту.

In [59]:
import h3
import branca.colormap as bcm
from shapely.geometry import Polygon

H3_RES = 9
print('середнє ребро комірки: %.0f м' % h3.average_hexagon_edge_length(H3_RES, unit='m'))

pc = cand.geometry.representative_point()
cand['h3'] = [h3.latlng_to_cell(p.y, p.x, H3_RES) for p in pc]          # (lat, lng) = (y, x)
dens = cand.groupby('h3').size().sort_values(ascending=False).rename('n')

# cell_to_boundary дає (lat, lng), а Polygon чекає (x, y) = (lng, lat) — міняємо місцями
hexes = gpd.GeoDataFrame(
    {'h3': dens.index, 'n': dens.values},
    geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in dens.index],
    crs='EPSG:4326')
assert hexes.intersects(poly).all(), 'шестикутники поза містом — переплутано порядок координат'

cum = dens.cumsum() / dens.sum()
K_HALF = int((cum < 0.5).sum() + 1)
N_HEX = len(dens)
print('кандидатів %d у %d комірках; найщільніша має %d' % (len(cand), N_HEX, dens.iloc[0]))
print('половина всіх кандидатів (%d з %d) — у %d комірках з %d (%.1f%% комірок)'
      % (int(np.ceil(len(cand) / 2)), len(cand), K_HALF, N_HEX, 100 * K_HALF / N_HEX))

cmap = bcm.linear.YlOrRd_09.scale(1, int(dens.max()))
cmap.caption = 'кандидатів у комірці H3 (рівень 9)'
c0 = station.geometry.iloc[0].representative_point()
fm_h3 = folium.Map(location=[c0.y, c0.x], zoom_start=13, tiles=BASEMAP)
folium.GeoJson(boundary, style_function=lambda f: {'color': '#444', 'weight': 2, 'fillOpacity': 0}).add_to(fm_h3)
folium.GeoJson(hexes[['h3', 'n', 'geometry']],
               style_function=lambda f: {'fillColor': cmap(f['properties']['n']), 'color': '#666',
                                         'weight': 0.4, 'fillOpacity': 0.65},
               tooltip=folium.GeoJsonTooltip(fields=['n'], aliases=['кандидатів:'])).add_to(fm_h3)
cmap.add_to(fm_h3)
_p30 = make_points(top_score)
for k, g in enumerate(_p30.geometry):
    folium.CircleMarker([g.y, g.x], radius=4, color='#1b5e20', fill=True, fill_opacity=1,
                        tooltip=('вокзал: ' if k == 0 else '%d. ' % k) + str(_p30['name'].iloc[k])).add_to(fm_h3)
fm_h3

середнє ребро комірки: 201 м
кандидатів 540 у 172 комірках; найщільніша має 31
половина всіх кандидатів (270 з 540) — у 19 комірках з 172 (11.0% комірок)


> **Пастка.** `h3.cell_to_boundary` повертає пари **(lat, lng)**, а
> `shapely.Polygon` чекає **(x, y)**, тобто (lng, lat). Переплутаєте — місто
> опиниться в Індійському океані. Це видно одразу, тому дивіться на карту.

### Питання до звіту

У скількох комірках зосереджено половину всіх пам'яток вашого міста?
Що це говорить про форму маршруту?

---
## 8. Просторове поєднання

### Завдання 8.1

Для кожної точки маршруту знайдіть найближчий заклад із шару `amenities`
(`sjoin_nearest`) і виведіть таблицю з відстанню в метрах.

Нагадування: `sjoin_nearest` рахує в одиницях CRS. У градусах результат буде
безглуздим — перепроєктуйте обидва шари в метричну систему.

In [60]:
# обидва шари — в метричну CRS ДО з’єднання, інакше відстань буде в градусах
am = amenities.copy()
am = am.set_geometry(am.geometry.representative_point()).to_crs(32635)
am['заклад'] = txt(am, 'name').where(txt(am, 'name') != '', '(без назви)')
am['тип'] = txt(am, 'amenity')
am = am[['заклад', 'тип', 'geometry']]

pm = pts10.to_crs(32635).copy()
pm['№ у матриці'] = range(len(pm))
assert pm.crs == am.crs and pm.crs.to_epsg() == 32635

near = gpd.sjoin_nearest(pm, am, how='left', distance_col='відстань, м')
near = near.drop_duplicates('№ у матриці').set_index('№ у матриці')     # при рівній відстані sjoin дає кілька рядків

order = ex10[:-1]                                                       # у порядку оптимального обходу
tab = pd.DataFrame({'крок': range(len(order)),
                    'точка маршруту': [pts10['name'].iloc[i] for i in order],
                    'найближчий заклад': [near.loc[i, 'заклад'] for i in order],
                    'тип': [near.loc[i, 'тип'] for i in order],
                    'відстань, м': [round(float(near.loc[i, 'відстань, м'])) for i in order]})
display(tab.set_index('крок'))
AM_MED = float(tab['відстань, м'].median()); AM_MAX = float(tab['відстань, м'].max())
print('медіана відстані до найближчого закладу: %.0f м, максимум: %.0f м' % (AM_MED, AM_MAX))

,точка маршруту,найближчий заклад,тип,"відстань, м"
крок,,,,
0,СТАРТ: Головний вокзал,Експрес,cafe,83
1,Національний музей у Львові,Da Vinci,restaurant,51
2,Львівський Державний Природознавчий Музей,Божевільний будинок,restaurant,40
3,Кам'яниця Корнякта,П'яте підземелля,restaurant,17
4,Львівський музей історії релігії,Cold comfort,cafe,20
5,Королівський арсенал,(без назви),toilets,38
6,Вежа Корнякта,Calcio,restaurant,25
7,Порохова Вежа,(без назви),toilets,82
8,"Арсенал, Львівський державний історичний музей...","Реберня ""Під Арсеналом""",restaurant,29


медіана відстані до найближчого закладу: 38 м, максимум: 83 м


---
## 9. Маршрут на карті

Лінія маршруту — це **геометрія ребер графа**, а не відрізки між пам'ятками.

### Завдання 9.1

Побудуйте карту, на якій **обидві** лінії: справжній маршрут по ребрах і
прямі відрізки між точками. Різниця має бути видно оком.

In [61]:
node_xy = nodes.set_index('osmid')[['x', 'y']]

# геометрія ребер у (lat, lon): ламана вулиця, а не пряма між двома вузлами
edge_ll = {}
for t in edges.itertuples():
    k = (t.u, t.v)
    if k not in edge_ll or float(t.length) < edge_ll[k][0]:
        edge_ll[k] = (float(t.length), [(y, x) for x, y in t.geometry.coords])


def route_nodes(tour, ids):
    """Послідовність вузлів графа для кожного відрізка маршруту."""
    legs = []
    for a, b in zip(tour[:-1], tour[1:]):
        try:
            legs.append(nx.shortest_path(G, ids[a], ids[b], weight='length'))
        except nx.NetworkXNoPath:
            legs.append(None)
    return legs


def route_coords(tour, ids):
    """Координати маршруту по ребрах графа у форматі (lat, lon) для folium."""
    coords = []
    for path in route_nodes(tour, ids):
        if path is None:
            continue
        if len(path) == 1:
            coords.append((float(node_xy.loc[path[0], 'y']), float(node_xy.loc[path[0], 'x'])))
        for u, v in zip(path[:-1], path[1:]):
            coords.extend(edge_ll[(u, v)][1] if (u, v) in edge_ll else
                          [(float(node_xy.loc[n, 'y']), float(node_xy.loc[n, 'x'])) for n in (u, v)])
    return coords


def route_length(tour, ids):
    return sum(G[u][v]['length'] for p in route_nodes(tour, ids) if p for u, v in zip(p[:-1], p[1:]))


def straight_coords(tour, pts):
    return [(pts.geometry.iloc[i].y, pts.geometry.iloc[i].x) for i in tour]


# перевірка: довжина намальованого маршруту збігається з довжиною з матриці
assert abs(route_length(ex10, ids10) - L_opt) < 1.0, 'маршрут на карті не збігається з матрицею'
assert abs(route_length(op30, ids30) - L_op30) < 1.0
L_straight10 = sum(float(E10[a, b]) for a, b in zip(ex10[:-1], ex10[1:]))
print('оптимальний маршрут: по ребрах графа %.2f км, по прямих відрізках %.2f км (різниця %.0f%%)'
      % (L_opt / 1000, L_straight10 / 1000, 100 * (L_opt / L_straight10 - 1)))


def badge(n, color):
    return folium.DivIcon(icon_size=(22, 22), icon_anchor=(11, 11), html=(
        '<div style="background:%s;color:#fff;border:2px solid #fff;border-radius:50%%;width:22px;'
        'height:22px;line-height:18px;text-align:center;font:bold 12px sans-serif;'
        'box-shadow:0 0 3px #000">%s</div>' % (color, n)))


c0 = pts10.geometry.iloc[0]
fmap = folium.Map(location=[c0.y, c0.x], zoom_start=15, tiles=BASEMAP)

g1 = folium.FeatureGroup(name='мала задача: по ребрах графа (оптимум Held–Karp)').add_to(fmap)
folium.PolyLine(route_coords(ex10, ids10), color='#1565c0', weight=5, opacity=0.9,
                tooltip='маршрут по ребрах: %.2f км' % (L_opt / 1000)).add_to(g1)
g2 = folium.FeatureGroup(name='мала задача: прямі відрізки між точками').add_to(fmap)
folium.PolyLine(straight_coords(ex10, pts10), color='#c62828', weight=3, opacity=0.9,
                dash_array='8 8', tooltip='прямі відрізки: %.2f км' % (L_straight10 / 1000)).add_to(g2)
g3 = folium.FeatureGroup(name='мала задача: точки').add_to(fmap)
for step, i in enumerate(ex10[:-1]):
    g = pts10.geometry.iloc[i]
    folium.Marker([g.y, g.x], icon=badge('С' if step == 0 else step, '#2e7d32' if step == 0 else '#1565c0'),
                  tooltip='%d. %s' % (step, pts10['name'].iloc[i])).add_to(g3)

g4 = folium.FeatureGroup(name='велика задача (30): 2-opt по ребрах', show=False).add_to(fmap)
folium.PolyLine(route_coords(op30, ids30), color='#6a1b9a', weight=4, opacity=0.85,
                tooltip='2-opt: %.2f км' % (L_op30 / 1000)).add_to(g4)
g5 = folium.FeatureGroup(name='велика задача (30): прямі відрізки', show=False).add_to(fmap)
folium.PolyLine(straight_coords(op30, pts30), color='#ef6c00', weight=2, dash_array='6 6').add_to(g5)
g6 = folium.FeatureGroup(name='велика задача (30): точки', show=False).add_to(fmap)
for step, i in enumerate(op30[:-1]):
    g = pts30.geometry.iloc[i]
    folium.Marker([g.y, g.x], icon=badge('С' if step == 0 else step, '#2e7d32' if step == 0 else '#6a1b9a'),
                  tooltip='%d. %s' % (step, pts30['name'].iloc[i])).add_to(g6)

folium.LayerControl(collapsed=False).add_to(fmap)
fmap.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:4px;left:50%;transform:translateX(-50%);z-index:9999;'
    'background:rgba(255,255,255,.85);padding:2px 8px;font:11px sans-serif">'
    'Дані © OpenStreetMap contributors, ODbL</div>'))
fmap.save('lr5_маршрут.html')
print('збережено lr5_маршрут.html (%.0f КБ)' % (os.path.getsize('lr5_маршрут.html') / 1024))
fmap

оптимальний маршрут: по ребрах графа 7.97 км, по прямих відрізках 6.48 км (різниця 23%)
збережено lr5_маршрут.html (128 КБ)


---
## 10. Результати для лабораторної роботи № 8

Збережіть файл, який ЛР8 прочитає і спробує побити.

In [62]:
result = {
    'студент': STUDENT_NAME,
    'номер': LIST_NUMBER,
    'місто': CITY,
    'джерело': '© OpenStreetMap contributors, ODbL',
    'мережа': 'walk',
    'старт': str(station['name'].iloc[0]),
    'мала_задача': {
        'точок': len(ids10),
        'оптимум_м': round(L_opt, 1),
        'оптимальний_порядок': [str(pts10['name'].iloc[i]) for i in ex10],
        'найближчий_сусід_м': round(tour_length(D10, nn10), 1),
        'двоопт_м': round(tour_length(D10, op10), 1),
        'оптимальний_порядок_індекси': [int(i) for i in ex10],
    },
    'велика_задача': {
        'точок': len(ids30),
        'найближчий_сусід_м': round(L_nn30, 1),
        'двоопт_м': round(L_op30, 1),
        'двоопт_с': round(t_op30, 4),
        'двоопт_порядок_індекси': [int(i) for i in op30],
        'точки': [{'name': str(pts30['name'].iloc[i]),
                   'lat': float(pts30.geometry.y.iloc[i]),
                   'lon': float(pts30.geometry.x.iloc[i])} for i in range(len(pts30))],
        'матриця_віддалей_м': [[None if not np.isfinite(v) else round(float(v), 1)
                                for v in row] for row in D30],
    },
}
with open('lr5_результати.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=1)
print('збережено lr5_результати.json (%.1f КБ)' % (os.path.getsize('lr5_результати.json') / 1024))

# самоперевірка файлу: ЛР8 прочитає його без нас
chk = json.load(open('lr5_результати.json', encoding='utf-8'))
M = chk['велика_задача']['матриця_віддалей_м']
assert len(M) == len(M[0]) == chk['велика_задача']['точок'] == len(chk['велика_задача']['точки'])
assert chk['мала_задача']['двоопт_м'] >= chk['мала_задача']['оптимум_м'] - 0.1
print('файл читається: матриця %dx%d, координат %d' % (len(M), len(M[0]), len(chk['велика_задача']['точки'])))


збережено lr5_результати.json (16.9 КБ)
файл читається: матриця 31x31, координат 31


Перед звітом збираємо числа в одному місці, щоб кожна відповідь спиралась на обчислене, а не на пам’ять.

In [63]:
# ---- числа для звіту: усе береться з обчислень вище ----
L_nn10, L_op10 = tour_length(D10, nn10), tour_length(D10, op10)
GAP_NN10 = 100 * (L_nn10 / L_opt - 1)
GAP_OP10 = 100 * (L_op10 / L_opt - 1)
GAP_NN30 = 100 * (L_nn30 / L_op30 - 1)
DF10, DF30 = detour_factor(D10, E10), detour_factor(D30, E30)
FIN10 = np.isfinite(D10)
ASYM10 = float(np.max(np.abs(np.where(FIN10 & FIN10.T, D10 - D10.T, 0.0))))
ASYM30 = float(np.max(np.abs(np.where(np.isfinite(D30) & np.isfinite(D30).T, D30 - D30.T, 0.0))))
N_INF10, N_INF30 = int(np.isinf(D10).sum()), int(np.isinf(D30).sum())
S10, S30 = SNAP['мала задача'], SNAP['велика задача']
MED_LEG30 = float(np.median([D30[a, b] for a, b in zip(op30[:-1], op30[1:])]))
NO_WIKI_NAIVE = int((top_naive['wiki'] == 0).sum())


def _js(o):
    if hasattr(o, 'item'):
        return o.item()
    if isinstance(o, (set, tuple)):
        return list(o)
    return str(o)


RD = {
    'student': STUDENT_NAME, 'city': CITY, 'list_number': LIST_NUMBER,
    'station': str(station['name'].iloc[0]),
    'N_RAW': N_RAW, 'N_BADTAG': N_BADTAG, 'N_NONAME': N_NONAME, 'N_CAND': N_CAND,
    'N_NONAME_WIKI': N_NONAME_WIKI, 'TOURISM_OK': TOURISM_OK,
    'geom': cand.geometry.geom_type.value_counts().to_dict(),
    'hist_top': cand['historic'].replace('', pd.NA).dropna().value_counts().head(3).to_dict()
                if 'historic' in cand.columns else {},
    'WEIGHTS': WEIGHTS, 'n_wiki': int(cand['wiki'].sum()), 'n_hist': int(cand['hist'].sum()),
    'n_visit': int(cand['visit'].sum()), 'n_lang_cols': len(lang_cols),
    'n_with_lang': int((cand['n_lang'] > 0).sum()),
    'overlap': overlap, 'overlap_raw': overlap_raw,
    'top10': list(top10['name']), 'naive10': list(top_naive['name']),
    'NO_WIKI_NAIVE': NO_WIKI_NAIVE,
    'naive_nowiki': list(top_naive[top_naive['wiki'] == 0]['name'].head(2)),
    'pairs_before': len(pairs), 'pairs_rows': PAIRS_DF.to_dict('records') if len(PAIRS_DF) else [],
    'dropped': [list(t) for t in dropped[:10]], 'N_DUP_DROPPED': N_DUP_DROPPED, 'pairs_left': len(left),
    'S10': S10, 'S30': S30, 'N_PED': N_PED,
    'DF10': DF10, 'DF30': DF30, 'N_INF10': N_INF10, 'N_INF30': N_INF30, 'ASYM10': ASYM10, 'ASYM30': ASYM30,
    'L_opt': L_opt, 'L_nn10': L_nn10, 'L_op10': L_op10, 'L_nn30': L_nn30, 'L_op30': L_op30,
    't_op30': t_op30, 'GAP_NN10': GAP_NN10, 'GAP_OP10': GAP_OP10, 'GAP_NN30': GAP_NN30,
    'L_straight10': L_straight10, 'MED_LEG30': MED_LEG30,
    'order10': [str(pts10['name'].iloc[i]) for i in ex10], 'ex10': ex10, 'op30': op30, 'nn30': nn30,
    'ids10': [int(x) for x in ids10], 'ids30': [int(x) for x in ids30],
    'HK_CELLS31': HK_CELLS31, 'HK_GB31': HK_GB31, 'HK_OPS31': HK_OPS31,
    'K_HALF': K_HALF, 'N_HEX': N_HEX, 'dens': {k: int(v) for k, v in dens.items()},
    'AM_MED': AM_MED, 'AM_MAX': AM_MAX, 'am_rows': tab.to_dict('records'),
    'SPEEDUP': SPEEDUP, 'T_LIVE': T_LIVE, 'T_SNAP': T_SNAP, 'MB_SAVED': MB_SAVED,
    'cost': COST, 'top30': list(top_score['name']),
}
with open('lr5_числа.json', 'w', encoding='utf-8') as f:
    json.dump(RD, f, ensure_ascii=False, indent=1, default=_js)
print('числа для звіту збережено: lr5_числа.json (%d полів)' % len(RD))

числа для звіту збережено: lr5_числа.json (70 полів)


---
## 11. Звіт

Відповіді пишіть просто в наступній комірці, числами зі своїх обчислень.

1. **Скільки об'єктів ви відкинули** на кроці 2 і за яким правилом? Чи погодились
   би ви віддати цей фільтр замовнику без пояснень?
2. **Які сигнали ввійшли у ваш рейтинг і з якими вагами?** Чому саме так?
3. **Скільки об'єктів збіглося** у ваших двох рейтингах із десяти? Що це
   означає для слова «найкращі»?
4. **Який у вас коефіцієнт обходу** і чи типовий він? Якщо ні — що в місті
   його пояснює?
5. **На скільки відсотків** найближчий сусід гірший за точний розв'язок?
   Чи здивувало вас це число?
6. **Чому точний розв'язок на 30 точках недоступний?** Назвіть конкретну
   величину, яка цьому заважає.
7. **Що у ваших даних виявилось неправильним** — дублікати, пропущені назви,
   об'єкти не на місці? Як ви з цим вчинили?

### Декларація про використання штучного інтелекту

Зазначте, чи користувались ШІ-помічником, у яких саме місцях і як перевіряли
отриманий код. Використання дозволене; **неперевірене** використання — ні.

In [64]:
def c(x, n=1):
    return (('%.' + str(n) + 'f') % x).replace('.', ',')


REPORT = f'''
Дані © OpenStreetMap contributors, ліцензія ODbL.

1. Відкинуто {N_BADTAG + N_NONAME} з {N_RAW}: {N_BADTAG} за тегами (правило 1), {N_NONAME} без name (правило 2; з них {N_NONAME_WIKI} мали wikidata/wikipedia). Лишилось {N_CAND}. Фільтр без пояснень замовнику не віддав би: безіменні об’єкти пропадають мовчки.
2. Сигнали й ваги: wikidata/wikipedia {WEIGHTS['wiki']}, переклади назви {WEIGHTS['lang_r']}, тип «для відвідування» {WEIGHTS['visit']}, historic {WEIGHTS['hist']}, площа {WEIGHTS['area_r']}. Стаття — єдиний зовнішній сигнал, тому найважча; площа найслабша.
3. У двох рейтингах збіглося {overlap} з {N_SMALL} об’єктів. «Найкращі» — властивість критерію, а не даних.
4. Коефіцієнт обходу: {c(DF10, 2)} (мала задача), {c(DF30, 2)} (велика); типовий діапазон 1,2–1,6.
5. Найближчий сусід гірший за точний розв’язок на {c(GAP_NN10)}%; 2-opt — на {c(GAP_OP10)}%.
6. Таблиця Held–Karp на 31 точці: {HK_CELLS31:.1e} клітин, {HK_GB31:.0f} ГБ — не вміщається в пам’ять.
7. Дублікатів вилучено {N_DUP_DROPPED}; на малій задачі {S10['точок']} точок зайняли {S10['вузлів']} вузлів, на великій {S30['точок']} — {S30['вузлів']}.

Декларація про ШІ: користувався Claude (Anthropic) для коду комірок «ВАШ КОД» і чернеток текстів; перевірки — у самому ноутбуці (метрична CRS, порядок координат, пішохідна мережа, check_snap, check_tour, збіг довжини маршруту на карті з матрицею). Повна версія — у Word-звіті.
'''
print(REPORT)


Дані © OpenStreetMap contributors, ліцензія ODbL.

1. Відкинуто 186 з 726: 0 за тегами (правило 1), 186 без name (правило 2; з них 9 мали wikidata/wikipedia). Лишилось 540. Фільтр без пояснень замовнику не віддав би: безіменні об’єкти пропадають мовчки.
2. Сигнали й ваги: wikidata/wikipedia 0.35, переклади назви 0.25, тип «для відвідування» 0.2, historic 0.1, площа 0.1. Стаття — єдиний зовнішній сигнал, тому найважча; площа найслабша.
3. У двох рейтингах збіглося 2 з 10 об’єктів. «Найкращі» — властивість критерію, а не даних.
4. Коефіцієнт обходу: 1,32 (мала задача), 1,22 (велика); типовий діапазон 1,2–1,6.
5. Найближчий сусід гірший за точний розв’язок на 6,8%; 2-opt — на 2,6%.
6. Таблиця Held–Karp на 31 точці: 3.2e+10 клітин, 258 ГБ — не вміщається в пам’ять.
7. Дублікатів вилучено 0; на малій задачі 11 точок зайняли 11 вузлів, на великій 31 — 30.

Декларація про ШІ: користувався Claude (Anthropic) для коду комірок «ВАШ КОД» і чернеток текстів; перевірки — у самому ноутбуці (метричн